In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import difflib

df = pd.read_csv(r"filename")

In [15]:
#Get counts of each Country in df
df['Country'].value_counts(dropna=False).sort_index()

Country
 Australia                     7
 Brazil                        1
 Canada                        6
 Germany                       3
 India                         8
 Ireland                       3
 United Kingdom                4
 United States                53
AU                             8
AUS                           13
Australia                    194
Australia                      7
BR                             9
Brasil                         9
Brazil                       136
Brazil                         4
CAN                           12
CANADA                        19
Canada                       274
Canada                         4
Country                        1
DE                            15
Deutschland                    6
England                       10
Germany                      261
Germany                        3
Great Britain                 10
IE                             8
INDIA                         22
India                        385
In

In [20]:
clean = df['Country'].str.strip().str.lower()
counts = clean.value_counts(dropna=False)
print(counts)

Country
united states               2948
india                        453
canada                       308
united kingdom               300
germany                      277
australia                    218
ireland                      182
brazil                       147
usa                           70
us                            67
u.s.                          65
united states of america      63
de                            15
aus                           13
can                           12
uk                            10
england                       10
great britain                 10
u.k.                           9
br                             9
brasil                         9
ie                             8
au                             8
republic of ireland            7
deutschland                    6
country                        1
NaN                            1
Name: count, dtype: int64


In [25]:
# 1. Normalize: strip spaces, lowercase
clean = df['Country'].str.strip().str.lower()

# 2. Create dictionary with correct country as the key, and aliases as values
aliases_by_country = {
    'australia': ['au', 'aus'],
    'brazil': ['br', 'brasil'],
    'canada': ['can'],
    'germany': ['de', 'deutschland'],
    'india': [],
    'ireland': [],
    'united kingdom': ['uk', 'gb', 'england'],
    'united states': ['us', 'usa', 'u.s.'],
}

# 3. Build the lookup tables from that one dictionary
canonical = list(aliases_by_country.keys())

aliases = {variant: correct
           for correct, variants in aliases_by_country.items()
           for variant in variants}

# 4. Standardize function (unchanged)
def standardize(value):
    if value in canonical:
        return value
    if value in aliases:
        return aliases[value]
    match = difflib.get_close_matches(value, canonical, n=1, cutoff=0.8)
    return match[0] if match else None

# 5. Map unique values, then apply to the column
mapping = {v: standardize(v) for v in clean.dropna().unique()}
df['Country_clean'] = clean.map(mapping).str.title()

# 6. Review what didn't match
unmatched = [v for v, m in mapping.items() if m is None]
print(unmatched)

# 7. Verify
print(df['Country_clean'].value_counts(dropna=False))

['republic of ireland', 'united states of america', 'u.k.', 'ie', 'great britain', 'country']
Country_clean
United States     3150
India              453
Canada             320
United Kingdom     320
Germany            298
Australia          239
Ireland            182
Brazil             165
NaN                 99
Name: count, dtype: int64


In [ ]:
#View results of what was cleaned from previous function
df.groupby('Country')['Country_clean'].value_counts()

Country          Country_clean 
 Australia       Australia            7
 Brazil          Brazil               1
 Canada          Canada               6
 Germany         Germany              3
 India           India                8
 Ireland         Ireland              3
 United Kingdom  United Kingdom       4
 United States   United States       53
AU               Australia            8
AUS              Australia           13
Australia        Australia          194
Australia        Australia            7
BR               Brazil               9
Brasil           Brazil               9
Brazil           Brazil             136
Brazil           Brazil               4
CAN              Canada              12
CANADA           Canada              19
Canada           Canada             274
Canada           Canada               4
DE               Germany             15
Deutschland      Germany              6
England          United Kingdom      10
Germany          Germany            261
Germany 